# 04 — Page extraction: clean -> extract -> confidence -> report

**What this turns into:** HTML/feed text -> one `Opportunity` per item, with

- title, organizer, type, location, eligibility, benefits, **deadline**
  (ISO `YYYY-MM-DD`), source URL, `source_checked` timestamp and a **confidence per field**;
- fields below `CONFIDENCE_THRESHOLD` — and any missing deadline — marked **"to verify"**;
- absent/rolling deadlines (`ongoing`, `present`, `current`) stored as **null**, never
  invented; unparseable claims become null + flagged;
- a failing page, feed or LLM call recorded as a failure — the run always finishes.

Works offline on `data/samples/*.html` + the sample feed (fake LLM). With real keys the
same cells run real Gemini **on the synthetic samples only**.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import get_settings, setup_logging
from oppradar_ai.ingestion import (
    PageFetcher,
    RunBudget,
    clean_html,
    extract_from_text,
    fields_filled,
    load_sources,
    parse_deadline,
    run_ingestion,
)
from oppradar_ai.schemas import Source

setup_logging('INFO')
settings = get_settings().ensure_dirs()
SAMPLES = settings.samples_dir

# Extraction uses whatever provider is configured (fake or gemini automatically).
# REAL_MODE additionally gates the one LIVE fetch in step 5 — real web text is
# displayed only and never sent to the LLM (privacy rule).
REAL_MODE = settings.llm_provider == 'gemini' and bool(settings.tavily_key_value)
print('LLM provider:', settings.llm_provider, '| live fetch:', 'yes' if REAL_MODE else 'no')

## Step 1 — Cleaning: raw HTML -> readable text

`clean_html` keeps the page title, the canonical permalink and the article body
(nav/scripts/cookie banners dropped) and collects outbound links. Garbage in ->
empty fields out, never an exception.

In [ ]:
pages = {}
for file in ('page_scholarship.html', 'page_hackathon.html'):
    pages[file] = clean_html((SAMPLES / file).read_text(encoding='utf-8'), str(SAMPLES / file))

clean_df = pd.DataFrame([
    {
        'file': file,
        'title': page.title[:52],
        'canonical url': page.canonical,
        'text chars': len(page.text),
        'links': len(page.links),
        'deadline in text': ('15 March 2027' in page.text) or ('2027-04-10' in page.text),
    }
    for file, page in pages.items()
])
display(clean_df)

## Step 2 — One budgeted extraction per page

One LLM call per page, counted against `MAX_EXTRACTIONS_PER_RUN` (default 20; disk-cache
hits are free). Rules decide everything the model may not guess. The confidence table
shows which fields are marked "to verify".

In [ ]:
budget = RunBudget(settings)
opps = []
for file, page in pages.items():
    opp = extract_from_text(
        page.text,
        url=page.canonical or str(SAMPLES / file),
        source_name='samples',
        title_hint=page.title,
        budget=budget,
        settings=settings,
    )
    if opp is not None:
        opps.append(opp)

fields_df = pd.DataFrame([
    {
        'title': o.title[:44],
        'organizer': o.organizer[:28],
        'type': o.type.value,
        'deadline (ISO)': str(o.deadline) if o.deadline else '(null - never invented)',
        'fields filled': fields_filled(o),
        'to verify': ', '.join(fc.field for fc in o.to_verify()) or '-',
        'url': o.url,
        'source_checked': o.source_checked.isoformat() if o.source_checked else '',
    }
    for o in opps
])
display(fields_df)
print('LLM usage for this step:', budget.extractions.model_dump())

if opps:
    conf_df = pd.DataFrame([
        {
            'field': fc.field,
            'confidence': fc.confidence,
            'status': 'TO VERIFY' if fc.needs_review else 'ok',
        }
        for fc in opps[0].field_confidences
    ])
    display(conf_df)

## Step 3 — Deadline normalization (ISO, never invented)

Raw text -> `date` or null. Rolling phrasing -> null; a month without a day -> null
(no guessing); impossible dates -> null; all-numeric dates read as day-first
(`11/30/2026` still resolves because 30 > 12).

In [ ]:
deadline_demo = pd.DataFrame([
    {'raw (as written)': '15 March 2027', 'ISO / null': parse_deadline('15 March 2027')},
    {'raw (as written)': '2027-04-10', 'ISO / null': parse_deadline('2027-04-10')},
    {'raw (as written)': '30/11/2026', 'ISO / null': parse_deadline('30/11/2026')},
    {'raw (as written)': '15 mars 2027', 'ISO / null': parse_deadline('15 mars 2027')},
    {'raw (as written)': 'ongoing', 'ISO / null': parse_deadline('ongoing') or '(null)'},
    {'raw (as written)': 'present', 'ISO / null': parse_deadline('present') or '(null)'},
    {'raw (as written)': 'March 2027 (no day)', 'ISO / null': parse_deadline('March 2027') or '(null)'},
    {'raw (as written)': 'sometime next year', 'ISO / null': parse_deadline('sometime next year') or '(null)'},
])
display(deadline_demo)

## Step 4 — A whole run: per-source table, usage and failures

`run_ingestion` over the two sample pages + the sample feed + one deliberately broken
path:

| what | where it lands |
|---|---|
| items found per source | `per_source[].fetched / extracted / new` |
| fields filled per source | `per_source[].fields_filled` |
| failures | `per_source[].errors` and `report.failures` |
| remote calls | `report.searches` / `report.extractions` (`made` / `cached` / `skipped`) |

The broken source proves the never-crash rule: logged, reported, zero exceptions.

In [ ]:
sample_sources = [
    Source(name='sample-feed', url=str(SAMPLES / 'feed_opportunities.xml'), kind='rss'),
    Source(name='sample-scholarship', url=str(SAMPLES / 'page_scholarship.html'), kind='page'),
    Source(name='sample-hackathon', url=str(SAMPLES / 'page_hackathon.html'), kind='page'),
    Source(name='broken-source', url=str(SAMPLES / 'missing_page.html'), kind='page'),
]
report = run_ingestion(sources=sample_sources, settings=settings)

per_source_df = pd.DataFrame([
    {
        'source': s.source,
        'items found': s.fetched,
        'extracted': s.extracted,
        'fields filled': s.fields_filled,
        'new': s.new,
        'failures': '; '.join(s.errors) or '-',
    }
    for s in report.per_source
])
display(per_source_df)

usage_df = pd.DataFrame([
    {'call kind': 'Tavily searches', **report.searches.model_dump()},
    {'call kind': 'LLM extractions', **report.extractions.model_dump()},
])
display(usage_df)

print(
    f'totals: fetched={report.items_fetched} new={report.items_new} '
    f'duplicates={report.items_duplicate} flagged={report.items_flagged}'
)
for failure in report.failures:
    print('failure:', failure)

## Step 5 — Live politeness check (real mode only)

One real fetch of a `sources.yaml` page through `PageFetcher` (User-Agent, delay,
robots.txt) — the cleaned text is **displayed only**: real web content is never sent
to the LLM. Offline this cell prints `skipped`.

In [ ]:
if REAL_MODE:
    target = next((s for s in load_sources() if s.enabled and s.kind == 'page'), None)
    if target is None:
        print('no enabled page source in sources.yaml')
    else:
        doc = PageFetcher(settings).get(target.url)
        if doc.ok:
            page = clean_html(doc.html, target.url)
            print(f'{target.name}: HTTP {doc.status}, {len(page.text)} chars cleaned')
            print('title:', page.title)
            print('(displayed only - real text is not sent to the LLM)')
        else:
            print(f'{target.name}: {doc.error} -> recorded as a failure, no crash')
else:
    print('skipped (OFFLINE mode): live fetch runs only with real keys')

## Expected output (offline / fake)

- **cleaning table**: 2 files, canonical URLs, 800+ chars each, deadline present;
- **fields table**: 2 opportunities with ISO deadlines or `(null - never invented)`,
  `to verify` listing the low-confidence fields, usage `made: 2` on a cold cache
  (or `cached: 2` free when an earlier run already extracted the samples);
- **confidence table**: 9 rows, `location` marked TO VERIFY on the fake output;
- **deadline demo**: dates parse to ISO; `ongoing` / `present` / `March 2027` /
  `sometime next year` show `(null)`;
- **report table**: feed = 3 items found, 2 pages = 1 each, `broken-source` shows
  `local file not found`; totals + failure lines printed;
- **usage**: step 4 counts the pages as `cached` (extracted in step 2, free) and
  a cold run counts the 3 feed entries as `made`; re-runs shift everything to
  `cached`; `made` can never exceed the per-run cap.